# Donut training ablation — quality

Field-level F1 from the `scripts/exp_*.sh` sweeps (one JSON per run in
`results/ablation/`, written by `train.py --ablation_out`).

- **F1 vs image size** — how much resolution training needs.
- **F1 / throughput vs batch size** — the most effective batch.

Join these with inference latency in `pareto.ipynb`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# One JSON per fine-tuning run, written by train.py --ablation_out via the
# scripts/exp_*.sh sweeps.
RESULTS = Path("../results/ablation")

rows = []
for p in sorted(RESULTS.glob("*.json")):
    d = json.loads(p.read_text())
    if "f1_strict" not in d:  # skip non-run JSONs
        continue
    h, w = d["image_size"]
    rows.append(
        {
            "tag": p.stem,
            "image_label": f"{h}x{w}",
            "image_px": h * w,
            "batch_size": d["batch_size"],
            "lr": d["lr"],
            "backend": d["backend"],
            "n_train": d["n_train"],
            "final_val_loss": d["final_val_loss"],
            "docs_per_sec": d["docs_per_sec"],
            "f1_strict": d["f1_strict"],
            "f1_soft": d["f1_soft"],
            "perfect": d["quality"]["doc_stats"]["perfect"],
            "n_with_gt": d["quality"]["n_with_gt"],
        }
    )

df = pd.DataFrame(rows)
print(f"loaded {len(df)} runs from {RESULTS}")
df

## F1 vs image size

In [ ]:
# F1 vs image size — locate the smallest resolution that still reads the document.
img = df[df.tag.str.startswith("imgsize_")].sort_values("image_px")
if img.empty:
    print("No imgsize_* runs — run scripts/exp_image_size.sh (DATA_JSON=... DEVICE=cuda).")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(img.image_px, img.f1_strict, marker="o", label="F1 strict")
    ax.plot(img.image_px, img.f1_soft, marker="s", label="F1 soft")
    for _, r in img.iterrows():
        ax.annotate(
            r.image_label,
            (r.image_px, (r.f1_strict if r.f1_strict is not None else 0)),
            textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8,
        )
    ax.set_xlabel("image pixels (H x W)")
    ax.set_ylabel("micro F1")
    ax.set_title("Field-extraction F1 vs image size")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()

## F1 and throughput vs batch size

In [ ]:
# F1 and training throughput vs batch size.
bat = df[df.tag.str.startswith("batch_")].sort_values("batch_size")
if bat.empty:
    print("No batch_* runs — run scripts/exp_batch_size.sh (DATA_JSON=... DEVICE=cuda).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(bat.batch_size, bat.f1_strict, marker="o", label="F1 strict")
    axes[0].plot(bat.batch_size, bat.f1_soft, marker="s", label="F1 soft")
    axes[0].set_ylabel("micro F1")
    axes[0].set_title("F1 vs batch size")
    axes[0].legend()
    axes[1].plot(bat.batch_size, bat.docs_per_sec, marker="o", color="tab:green")
    axes[1].set_ylabel("train docs / s")
    axes[1].set_title("Training throughput vs batch size")
    for ax in axes:
        ax.set_xlabel("batch size")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## All runs ranked by F1

In [ ]:
# All runs ranked by strict F1. Top row = best field extraction so far.
cols = [
    "tag", "image_label", "batch_size", "f1_strict", "f1_soft",
    "perfect", "n_with_gt", "final_val_loss", "docs_per_sec",
]
df.sort_values("f1_strict", ascending=False, na_position="last")[cols]